# SpendShield — Revised Dataset Validation

This notebook validates version `v2`, builds its candidate feature matrix, and
checks behavioral overlap without using scenario metadata as model input.


## Validation boundary

The source is read from `data/synthetic/v2/`. Historical features are audited
with timestamp plus transaction-ID ordering; current rows are excluded from
their own history. The generated labels remain synthetic-only.


In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "ml").is_dir() and (candidate / "data" / "synthetic").is_dir():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

V2_DIR = ROOT / "data" / "synthetic" / "v2"
V2_FEATURE_DIR = V2_DIR / "features"
V2_BASELINE_DIR = V2_DIR / "baseline"
V2_ERROR_DIR = V2_DIR / "error_analysis"
V2_COMPARISON_DIR = ROOT / "data" / "synthetic" / "comparison"
from ml.synthetic_dataset_generator import validate_written_dataset
from ml.feature_matrix import build_feature_artifact, validate_feature_artifact

dataset_validation = validate_written_dataset(V2_DIR)
feature_manifest = build_feature_artifact(V2_DIR, V2_FEATURE_DIR)
feature_validation = validate_feature_artifact(V2_FEATURE_DIR)
print(json.dumps({
    "dataset_valid": dataset_validation["valid"],
    "feature_valid": feature_validation["valid"],
    "feature_count": feature_validation["feature_count"],
    "feature_names": feature_validation["feature_names"],
    "split_counts": feature_validation["split_counts"],
    "excluded_features": feature_validation["excluded_fields_present_as_features"],
}, indent=2))


{
  "dataset_valid": true,
  "feature_valid": true,
  "feature_count": 32,
  "feature_names": [
    "amount",
    "transaction_hour",
    "day_of_week",
    "user_historical_transaction_count_before",
    "user_historical_average_amount_before",
    "time_since_previous_transaction_seconds",
    "user_historical_category_frequency_before",
    "merchant_novelty_before",
    "channel_novelty_before",
    "user_relative_amount_deviation",
    "user_relative_time_deviation",
    "time_since_previous_transaction_seconds__missing",
    "currency__INR",
    "currency____unknown",
    "merchant_category____unknown",
    "merchant_category__bills",
    "merchant_category__education",
    "merchant_category__entertainment",
    "merchant_category__food",
    "merchant_category__grocery",
    "merchant_category__healthcare",
    "merchant_category__home",
    "merchant_category__other",
    "merchant_category__shopping",
    "merchant_category__subscriptions",
    "merchant_category__transport",

In [2]:
import csv

with (V2_DIR / "validation.csv").open("r", encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))

def rate(label, predicate):
    selected = [row for row in rows if row["scenario_label"] == label]
    return round(sum(predicate(row) for row in selected) / len(selected), 6) if selected else 0.0

print({
    "rapid_repeat_le_600_seconds": rate("synthetic_rapid_repeat", lambda row: row["time_since_previous_transaction_seconds"] not in ("", None) and int(float(row["time_since_previous_transaction_seconds"])) <= 600),
    "unusual_time_global_window": rate("synthetic_unusual_time", lambda row: int(row["transaction_hour"]) in {0, 1, 2, 3, 4, 23}),
    "combined_rows_global_window": rate("synthetic_combined_pattern", lambda row: int(row["transaction_hour"]) in {0, 1, 2, 3, 4, 23}),
    "normal_late_hour_overlap": rate("normal", lambda row: int(row["transaction_hour"]) in {0, 1, 2, 3, 4, 23}),
})


{'rapid_repeat_le_600_seconds': 0.233871, 'unusual_time_global_window': 0.333333, 'combined_rows_global_window': 0.166667, 'normal_late_hour_overlap': 0.246646}


The overlap rates are diagnostics of the generator, not proof of
real-world behavior or detector quality. Any failed leakage or schema check
blocks the next research step.
